# P2-A4 — RAG End-to-End (retrieve + generate)

Assembly time. You have all three pieces: **prompting** (P2-A1), **structured/grounded calls**, and **semantic retrieval** (P2-A3). RAG = retrieve the relevant context, stuff it into the prompt, and let the LLM answer *from that context*.

The knowledge base below is about a **made-up company ("Zephyr Cloud")** — facts the base model cannot possibly know from training. That's deliberate: if the model answers correctly, it can *only* be because your retrieval worked. It's how you prove RAG is actually grounding the answer.

In [ ]:
# --- Setup (given) ---
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
CHAT_MODEL = 'gpt-4o-mini'
EMBED_MODEL = 'text-embedding-3-small'

def embed(texts):
    if isinstance(texts, str):
        texts = [texts]
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data])

def ask(prompt, system=None, **kwargs):
    messages = []
    if system:
        messages.append({'role': 'system', 'content': system})
    messages.append({'role': 'user', 'content': prompt})
    resp = client.chat.completions.create(model=CHAT_MODEL, messages=messages, **kwargs)
    return resp.choices[0].message.content

# --- Given: knowledge base about a FICTIONAL company (model can't know these) ---
kb = [
    "Zephyr Cloud's free tier includes 5 GB of storage and 100 GB of bandwidth per month.",
    "To upgrade your Zephyr plan, go to Billing > Plans and choose Pro or Enterprise.",
    "Zephyr Cloud's Pro plan costs $29 per month and includes 1 TB of storage.",
    "Zephyr support is available 24/7 via in-app chat for Pro and Enterprise customers.",
    "Zephyr's data centers are located in Oregon, Frankfurt, and Singapore.",
]
kb_emb = embed(kb)   # embed the knowledge base once, up front
print('KB embedded:', kb_emb.shape)

## Task 1 — `retrieve(question, k)`
Write a function that embeds the question, cosine-compares it to `kb_emb`, and returns the **top-k** most relevant KB strings.
<br>This is your P2-A3 code wrapped in a function. Test it: `retrieve('how much is Pro?', k=2)` should surface the $29 fact.

In [ ]:
# TODO: def retrieve(question, k=2): ... return the top-k kb strings


## Task 2 — `answer(question)` = retrieve + generate
Write the full RAG function: call `retrieve()`, build a prompt that includes the retrieved context, and instruct the model to **answer ONLY from that context — and say it doesn't know if the answer isn't there.** Return the model's answer.
<br>Hint: your prompt should look roughly like: `Context:\n{joined retrieved docs}\n\nQuestion: {question}\n\nAnswer using only the context above; if it's not there, say you don't know.`

In [ ]:
# TODO: def answer(question): retrieve context, build grounded prompt, return ask(...)


## Task 3 — Test grounding (the moment of truth)
Call `answer()` on two questions and print both:
1. **Answerable** from the KB — e.g. *"How much does the Pro plan cost and how much storage does it include?"* → should give the right facts.
2. **NOT in the KB** — e.g. *"Does Zephyr have a data center in Tokyo?"* → the model should say it doesn't know / it's not in the context (NOT make something up).
<br>Goal: prove RAG both *answers from your data* and *refuses to hallucinate* when the data isn't there.

In [ ]:
# TODO: test answer() on an in-KB question and an out-of-KB question


## Task 4 — Explain (in your own words)
1. Why does RAG let a model correctly answer about 'Zephyr Cloud' when it was never trained on it? What problem does that solve for real apps?
2. What is the single biggest failure point of this pipeline — i.e. if the final answer is wrong, where did it most likely break, and how would you measure that (tie it back to evals, W2-A2)?

> _your answer here_